# Develop Ensemble of ML Models to Predict SWE

This notebook will train and evaluate multiple machine learning models to compare algorithms and develop an ensemble mean value of predicted SWE across the spatial grid.

## Step 1: Set up notebook

In [84]:
# import libraries

# file management
import joblib
import os
import pathlib
from pathlib import Path
#import pickle
import pyarrow.parquet as pq

# datatypes
import numpy as np
import pandas as pd
import xarray as xr

# geospatial data
import geopandas as gpd
import rioxarray as rxr

# machine learning
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, PredefinedSplit, RandomizedSearchCV, TimeSeriesSplit
import xgboost as xgb

# plotting
import holoviews as hv
import hvplot.pandas
import hvplot.xarray
import matplotlib
import matplotlib.pyplot as plt
import panel as pn

# misc
import math

In [5]:
# set directories
proj_dir = os.path.join(pathlib.Path.home(),
                        'Documents',
                        'Graduate_School',
                        'EDA_Certificate', 
                        'Summer', 
                        'snow-drought-modeling')

raw_data_dir = os.path.join(proj_dir, 'data', 'raw')
cleaned_data_dir = os.path.join(proj_dir, 'data', 'cleaned')

In [6]:
# set file paths

# machine learning
ml_dir = Path(cleaned_data_dir, 'machine_learning')
train_path = Path(ml_dir, 'stations_train.parquet')
test_path = Path(ml_dir, 'stations_test.parquet')

# outputs
output_dir = Path(proj_dir, 'outputs')

In [7]:
# import PRSIM training dataset
if os.path.exists(train_path):
    # import
    stations_train = pd.read_parquet(train_path)
    # print success
    print("Training data successfully imported!")
else:
    print(f"Training data not found. Check {ml_dir} for file.")

if os.path.exists(test_path):
    stations_test = pd.read_parquet(test_path)
    print("Testing data successfully imported!")
else:
    print(f"Test data not found. Check {ml_dir} for file.")

Training data successfully imported!
Testing data successfully imported!


In [8]:
# check out training dataset
stations_train

,date,stationTriplet,latitude,longitude,swe_mm,prism_ppt_mm,prism_tmin_c,prism_tmax_c,elevation,slope,...,prism_tmax_c_roll_3d,prism_ppt_mm_roll_3d,prism_tmin_c_roll_7d,prism_tmax_c_roll_7d,prism_ppt_mm_roll_7d,water_year,prism_tmean_c,wy_cumul_precip_mm,wy_cumul_melt_days,dowy
0,1997-10-01,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,3.541,15.947000,2996,21.693270,...,15.947000,0.000,3.541000,15.947000,0.000,1998,9.743999,0.000000,9.743999,1
1,1997-10-02,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,6.940,16.136000,2996,21.693270,...,16.041500,0.000,5.240500,16.041500,0.000,1998,11.538000,0.000000,21.282000,2
2,1997-10-03,318:MT:SNTL,44.47147,-112.98191,0.0,5.717,-2.346,9.064000,2996,21.693270,...,13.715666,5.717,2.711667,13.715666,5.717,1998,3.359000,5.717000,24.640999,3
3,1997-10-04,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,-2.784,8.394000,2996,21.693270,...,11.198000,5.717,1.337750,12.385250,5.717,1998,2.805000,5.717000,27.445999,4
4,1997-10-05,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,-2.662,8.978000,2996,21.693270,...,8.812000,5.717,0.537800,11.703800,5.717,1998,3.158000,5.717000,30.604000,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113066,2015-05-28,924:MT:SNTL,44.65866,-111.09199,0.0,4.934,4.412,16.138000,2035,1.305896,...,15.328000,12.286,2.044857,14.711000,20.626,2015,10.275001,369.302002,-334.210510,240
113067,2015-05-29,924:MT:SNTL,44.65866,-111.09199,0.0,4.307,1.766,11.901000,2035,1.305896,...,13.822667,13.262,2.063143,14.538000,22.514,2015,6.833500,373.609009,-327.376984,241
113068,2015-05-30,924:MT:SNTL,44.65866,-111.09199,0.0,0.038,-0.797,19.499001,2035,1.305896,...,15.846000,9.279,2.055000,15.185286,21.806,2015,9.351000,373.647003,-318.026001,242
113069,2015-05-31,924:MT:SNTL,44.65866,-111.09199,0.0,5.639,1.194,20.554001,2035,1.305896,...,17.318000,9.984,2.228714,16.117000,24.702,2015,10.874001,379.286011,-307.152008,243


In [6]:
# check out testing dataset
stations_test

,date,stationTriplet,latitude,longitude,swe_mm,prism_ppt_mm,prism_tmin_c,prism_tmax_c,elevation,slope,...,prism_tmax_c_roll_3d,prism_ppt_mm_roll_3d,prism_tmin_c_roll_7d,prism_tmax_c_roll_7d,prism_ppt_mm_roll_7d,water_year,prism_tmean_c,wy_cumul_precip_mm,wy_cumul_melt_days,dowy
0,2015-10-01,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,4.465,18.061001,2996,21.693270,...,18.061001,0.000,4.465000,18.061001,0.000,2016,11.263000,0.000000,11.263000,1
1,2015-10-02,318:MT:SNTL,44.47147,-112.98191,0.0,4.653,3.988,13.879000,2996,21.693270,...,15.970000,4.653,4.226500,15.970000,4.653,2016,8.933500,4.653000,20.196501,2
2,2015-10-03,318:MT:SNTL,44.47147,-112.98191,0.0,10.110,0.908,6.044000,2996,21.693270,...,12.661334,14.763,3.120333,12.661334,14.763,2016,3.476000,14.763000,23.672501,3
3,2015-10-04,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,-4.195,2.711000,2996,21.693270,...,7.544667,14.763,1.291500,10.173750,14.763,2016,-0.742000,14.763000,22.930500,4
4,2015-10-05,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,-4.515,5.482000,2996,21.693270,...,4.745667,10.110,0.130200,9.235400,14.763,2016,0.483500,14.763000,23.414001,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30545,2020-05-28,924:MT:SNTL,44.65866,-111.09199,0.0,0.000,1.235,21.813000,2035,1.305896,...,18.629333,0.765,0.284714,12.022286,33.836,2020,11.524000,360.126007,-942.330505,241
30546,2020-05-29,924:MT:SNTL,44.65866,-111.09199,0.0,0.000,0.026,24.500999,2035,1.305896,...,21.185666,0.765,0.276143,14.380286,18.482,2020,12.263499,360.126007,-930.067017,242
30547,2020-05-30,924:MT:SNTL,44.65866,-111.09199,0.0,0.726,1.743,27.243999,2035,1.305896,...,24.519333,0.726,0.532857,17.315143,12.930,2020,14.493500,360.851990,-915.573486,243
30548,2020-05-31,924:MT:SNTL,44.65866,-111.09199,0.0,0.480,3.636,28.344999,2035,1.305896,...,26.696666,1.206,1.157286,20.901143,2.620,2020,15.990499,361.332001,-899.583008,244


In [9]:
# rename PRISM training dataset var names to be clean

# training
stations_train.rename(columns = {
    'prism_ppt_mm': 'ppt_mm', 
    'prism_tmin_c': 'tmin_c', 'prism_tmax_c': 'tmax_c', 'prism_tmean_c': 'tmean_c',
    'prism_tmin_c_roll_3d': 'tmin_c_roll_3d', 'prism_tmax_c_roll_3d': 'tmax_c_roll_3d', 'prism_ppt_mm_roll_3d': 'ppt_mm_roll_3d',
    'prism_tmin_c_roll_7d': 'tmin_c_roll_7d', 'prism_tmax_c_roll_7d': 'tmax_c_roll_7d', 'prism_ppt_mm_roll_7d': 'ppt_mm_roll_7d',
    }, inplace=True)

# testing
stations_test.rename(columns = {
    'prism_ppt_mm': 'ppt_mm', 
    'prism_tmin_c': 'tmin_c', 'prism_tmax_c': 'tmax_c', 'prism_tmean_c': 'tmean_c',
    'prism_tmin_c_roll_3d': 'tmin_c_roll_3d', 'prism_tmax_c_roll_3d': 'tmax_c_roll_3d', 'prism_ppt_mm_roll_3d': 'ppt_mm_roll_3d',
    'prism_tmin_c_roll_7d': 'tmin_c_roll_7d', 'prism_tmax_c_roll_7d': 'tmax_c_roll_7d', 'prism_ppt_mm_roll_7d': 'ppt_mm_roll_7d',
    }, inplace=True)

In [10]:
# check that worked
stations_train

,date,stationTriplet,latitude,longitude,swe_mm,ppt_mm,tmin_c,tmax_c,elevation,slope,...,tmax_c_roll_3d,ppt_mm_roll_3d,tmin_c_roll_7d,tmax_c_roll_7d,ppt_mm_roll_7d,water_year,tmean_c,wy_cumul_precip_mm,wy_cumul_melt_days,dowy
0,1997-10-01,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,3.541,15.947000,2996,21.693270,...,15.947000,0.000,3.541000,15.947000,0.000,1998,9.743999,0.000000,9.743999,1
1,1997-10-02,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,6.940,16.136000,2996,21.693270,...,16.041500,0.000,5.240500,16.041500,0.000,1998,11.538000,0.000000,21.282000,2
2,1997-10-03,318:MT:SNTL,44.47147,-112.98191,0.0,5.717,-2.346,9.064000,2996,21.693270,...,13.715666,5.717,2.711667,13.715666,5.717,1998,3.359000,5.717000,24.640999,3
3,1997-10-04,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,-2.784,8.394000,2996,21.693270,...,11.198000,5.717,1.337750,12.385250,5.717,1998,2.805000,5.717000,27.445999,4
4,1997-10-05,318:MT:SNTL,44.47147,-112.98191,0.0,0.000,-2.662,8.978000,2996,21.693270,...,8.812000,5.717,0.537800,11.703800,5.717,1998,3.158000,5.717000,30.604000,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113066,2015-05-28,924:MT:SNTL,44.65866,-111.09199,0.0,4.934,4.412,16.138000,2035,1.305896,...,15.328000,12.286,2.044857,14.711000,20.626,2015,10.275001,369.302002,-334.210510,240
113067,2015-05-29,924:MT:SNTL,44.65866,-111.09199,0.0,4.307,1.766,11.901000,2035,1.305896,...,13.822667,13.262,2.063143,14.538000,22.514,2015,6.833500,373.609009,-327.376984,241
113068,2015-05-30,924:MT:SNTL,44.65866,-111.09199,0.0,0.038,-0.797,19.499001,2035,1.305896,...,15.846000,9.279,2.055000,15.185286,21.806,2015,9.351000,373.647003,-318.026001,242
113069,2015-05-31,924:MT:SNTL,44.65866,-111.09199,0.0,5.639,1.194,20.554001,2035,1.305896,...,17.318000,9.984,2.228714,16.117000,24.702,2015,10.874001,379.286011,-307.152008,243


In [16]:
# split training and validation datasets

# get list of water years
train_wy_unique = sorted(stations_train['water_year'].unique())
# get year value hold out last 3 years of training block
val_start_year = train_wy_unique[-3]

# subset the datasets
stations_train_subset = stations_train[stations_train['water_year'] < val_start_year]
stations_val = stations_train[stations_train['water_year'] >= val_start_year]

In [17]:
# split training data
x_train = stations_train_subset[[
    # lat and lon
    'latitude', 'longitude', 
    # raw prism data
    'ppt_mm', 'tmin_c', 'tmax_c', 
    # topographic data
    'elevation', 'slope', 'aspect_north', 'aspect_east', 
    # lagged prism variables 7-day (drop lagged prism variables 3-day)
    'tmin_c_roll_7d', 'tmax_c_roll_7d', 'ppt_mm_roll_7d',
    # cumulative precip and positive degree days
    'wy_cumul_precip_mm', 'wy_cumul_melt_days',
    # day of water year
    'dowy']]
y_train = stations_train_subset[['swe_mm']]

# split validation data
x_val = stations_val[[
    'latitude', 'longitude', 
    'ppt_mm', 'tmin_c', 'tmax_c', 
    'elevation', 'slope', 'aspect_north', 'aspect_east', 
    'tmin_c_roll_7d', 'tmax_c_roll_7d', 'ppt_mm_roll_7d',
    'wy_cumul_precip_mm', 'wy_cumul_melt_days',
    'dowy']]
y_val = stations_val[['swe_mm']]

# split testing data
x_test = stations_test[[
    'latitude', 'longitude', 
        'ppt_mm', 'tmin_c', 'tmax_c', 
        'elevation', 'slope', 'aspect_north', 'aspect_east', 
        'tmin_c_roll_7d', 'tmax_c_roll_7d', 'ppt_mm_roll_7d',
        'wy_cumul_precip_mm', 'wy_cumul_melt_days',
        'dowy']]
y_test = stations_test[['swe_mm']]

In [ ]:
# create tuning dataFrame with markers for test and validation split
# I used Gemini to help me with the cross-validation split
x_tuning = pd.concat([x_train, x_val], axis=0).reset_index(drop=True)
y_tuning = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

# mark -1 for training indices, 0 for validation indices
split_indices = [-1] * len(x_train) + [0] * len(x_val)

In [18]:
# check that SWE is in y_train and y_test, not x_train and x_test
# make dict of dfs
dfs = {
    'x_train': x_train,
    'y_train': y_train,
    'x_val': x_val,
    'y_val': y_val,
    'x_test': x_test,
    'y_test': y_test
}

# check that swe is in right places in the dict
for name, df in dfs.items():
    if 'swe_mm' in df.columns:
        print(f"SWE found in {name}")
    else:
        print(f"SWE not found in {name}")

SWE not found in x_train
SWE found in y_train
SWE not found in x_val
SWE found in y_val
SWE not found in x_test
SWE found in y_test


## Step 2: Retrain Random Forests Model

Use final hyperparameter config from 05

In [63]:
# initialize RF model with final parameters from 05 notebook

rf_final = RandomForestRegressor(
    n_estimators=400,
    max_features='sqrt',
    max_depth=40,
    random_state=42,
    n_jobs=2
)

In [65]:
# fit model
rf_final.fit(x_tuning, y_tuning)

c:\Users\raini\miniconda3\envs\earth-analytics-python\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


Tuned Model:
R²: 0.7365587997438342
RMSE: 116.06029514044268 mm


In [66]:
# print metrics
print('Tuned Model:')
print('R²:', rf_final.score(x_test, y_test))
print(f'RMSE: {root_mean_squared_error(y_test, rf_final.predict(x_test))} mm')

Tuned Model:
R²: 0.7365587997438342
RMSE: 116.06029514044268 mm


In [ ]:
# predict
rf_preds = rf_final.predict(x_test)

## Step 3: Train XGBoost Model

Per Alabi et al. 2026 and O'Flaherty 2025, XGBoost is a common model candidate. Alabi et al. 2026 also highlights the performance of LightGBM, so I'll use that as well.

In [ ]:
# initialize XGBoost

xgb_base = xgb.XGBRegressor(
    # use a histogram-based tree method
    tree_method = 'hist',
    # set early stopping rounds to improve model and tuning speed
    early_stopping_rounds = 25, 
    # set the random state for repeatability
    random_state = 42,
    # set n jobs to prioritize CPU usage
    n_jobs = -1
)

In [29]:
# train base XGBoost model
xgb_base.fit(x_train, y_train, eval_set=[(x_val, y_val)], verbose=0)

# print some metrics
print('Baseline Model:')
print('R^2:', xgb_base.score(x_test, y_test))
print(f'RMSE: {root_mean_squared_error(y_test, xgb_base.predict(x_test))} mm')
print("Stopped on iteration: ", xgb_base.best_iteration)

Baseline Model:
R^2: 0.7285417914390564
RMSE: 117.81302642822266 mm
Stopped on iteration:  8


In [ ]:
# xgboost hyperparameter tuning

# set predefined split
pds = PredefinedSplit(test_fold=split_indices)

# set param grid
param_grid = {
    # this sets how quickly the model improves - higher numbers run faster but risk overfitting
    'learning_rate': [0.01, 0.05, 0.1],
    # 
    'max_depth': [4, 6, 8],
    # force model to explore various relationships between features, vs just relying on dowy or lat/lon
    'colsample_bytree': [0.6, 0.8],
    # control how much of the dataset is sampled at a timestep
    'subsample': [0.7, 0.85, 1]
}

In [27]:
# initialize randomized search
xgb_search = RandomizedSearchCV(
    estimator = xgb.XGBRegressor(
        n_estimators=500,
        early_stopping_rounds=25,
        tree_method='hist',
        random_state=42,
        n_jobs=-1),
        
    # set parameter grid to use
    param_distributions=param_grid,
    # set number of iterations to test
    n_iter = 15,
    # set cross validation split
    cv = pds,
    # set the scoring method to negative RMSE to penalize large errors
    scoring = 'neg_root_mean_squared_error',
    random_state = 42
)

In [28]:
# fit search model
xgb_search.fit(x_tuning, y_tuning, eval_set = [(x_val, y_val)], verbose=0)

Best Parameters: {'subsample': 0.7, 'max_depth': 8, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
Holdout Test RMSE: 116.45 mm
Holdout Test R²: 0.7348


In [40]:
# print evaluation metrics
best_xgb = xgb_search.best_estimator_
print("Best Parameters:", xgb_search.best_params_)
print(f"RMSE: {root_mean_squared_error(y_test, best_xgb.predict(x_test))} mm")
print(f"R²: {r2_score(y_test, best_xgb.predict(x_test))}")
#print("Stopped on iteration: ", xgb_search.best_iteration_)

Best Parameters: {'subsample': 0.7, 'max_depth': 8, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
RMSE: 116.45013427734375 mm
R²: 0.7347860336303711


In [75]:
# run final XGBoost model
xgb_final = xgb.XGBRegressor(
    # parameters from tuning
    subsample = 0.7,
    max_depth = 8,
    learning_rate = 0.01,
    colsample_bytree = 0.6,
    n_estimators = 500,

    # use some parameters from base model
    tree_method = 'hist',
    early_stopping_rounds = 25, 
    random_state = 42,
    n_jobs = -1
)

# fit model
xgb_final.fit(x_tuning, y_tuning, eval_set = [(x_val, y_val)], verbose=0)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.6
,device,None
,early_stopping_rounds,25
,enable_categorical,False
,eval_metric,None


In [76]:
# print final statistics
print('Baseline Model:')
print('R^2:', xgb_base.score(x_test, y_test))
print(f'RMSE: {root_mean_squared_error(y_test, xgb_base.predict(x_test))} mm')
print("Stopped on iteration: ", xgb_base.best_iteration)

print('Tuned Model:')
print('R^2:', xgb_final.score(x_test, y_test))
print(f'RMSE: {root_mean_squared_error(y_test, xgb_final.predict(x_test))} mm')
print("Stopped on iteration: ", xgb_final.best_iteration)

Baseline Model:
R^2: 0.7285417914390564
RMSE: 117.81302642822266 mm
Stopped on iteration:  8
Tuned Model:
R^2: 0.7347860336303711
RMSE: 116.45013427734375 mm
Stopped on iteration:  499


## Step 4: Train LightGBM Model

In [48]:
# set a base model
lgb_base = lgb.LGBMRegressor(
    n_estimators = 700,
    random_state = 42,
    verbose = -1,
    n_jobs = -1
)

# fit base model
lgb_base.fit(x_train, y_train, 
             eval_X = x_val, 
             eval_y = y_val,
             eval_metric='rmse',
             callbacks=[
                # add early stopping
                lgb.early_stopping(stopping_rounds=25, verbose=False), 
                lgb.log_evaluation(period=0)])

# print metrics
print('Baseline LightGBM Model:')
print('R²:', lgb_base.score(x_test, y_test))
print(f'RMSE: {root_mean_squared_error(y_test, lgb_base.predict(x_test))} mm')
print("Stopped on iteration:", lgb_base.best_iteration_)

Baseline LightGBM Model:
R²: 0.7308460886136348
RMSE: 117.31192739575243 mm
Stopped on iteration: 40


In [49]:
# set parameter distribution for hyperparameter tuning
param_grid = {
    'learning_rate': [0.01, 0.03, 0.05],
    'num_leaves': [15, 25, 31],
    'max_depth': [4, 6, 8],
    'subsample': [0.7, 0.85, 1.0],
    'colsample_bytree': [0.6, 0.8, 1]
}

In [50]:
# initialize randomized search
lgb_search = RandomizedSearchCV(
    estimator = lgb.LGBMRegressor(
        n_estimators=750,
        # activate row subsampling
        subsample_freq=1,
        random_state=42,
        n_jobs=-1,
        verbose = -1),
        
    # set parameter grid to use
    param_distributions=param_grid,
    # set number of iterations to test
    n_iter = 15,
    # set cross validation split
    cv = pds,
    # set the scoring method to negative RMSE to penalize large errors
    scoring = 'neg_root_mean_squared_error',
    # prevent model from retraining on validation data
    refit = False,
    random_state = 42,
)

In [51]:
# fit search
lgb_search.fit(
    x_tuning, y_tuning,
    eval_X = x_val, 
    eval_y = y_val,
    eval_metric = 'rmse',
    callbacks=[
            lgb.early_stopping(stopping_rounds=25, verbose=False),
            lgb.log_evaluation(period=0)]
)

,estimator,LGBMRegressor... verbose=-1)
,param_distributions,"{'colsample_bytree': [0.6, 0.8, ...], 'learning_rate': [0.01, 0.03, ...], 'max_depth': [4, 6, ...], 'num_leaves': [15, 25, ...], ...}"
,n_iter,15
,scoring,'neg_root_mean_squared_error'
,n_jobs,None
,refit,False
,cv,"PredefinedSpl...pe=(113071,)))"
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [53]:
print("Best Parameters:", lgb_search.best_params_)

Best Parameters: {'subsample': 0.7, 'num_leaves': 31, 'max_depth': 8, 'learning_rate': 0.01, 'colsample_bytree': 0.6}


In [54]:
# take best params
best_lgb = lgb.LGBMRegressor(
    **lgb_search.best_params_,
    n_estimators=750,
    subsample_freq=1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# refit model
best_lgb.fit(
    x_tuning, y_tuning,
    eval_X = x_val, 
    eval_y = y_val,
    eval_metric = 'rmse',
    callbacks=[
            lgb.early_stopping(stopping_rounds=25, verbose=False),
            lgb.log_evaluation(period=0)]
)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,8
,learning_rate,0.01
,n_estimators,750
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [55]:
# print evaluation metrics
print("Best Parameters:", lgb_search.best_params_)
print(f"RMSE: {root_mean_squared_error(y_test, best_lgb.predict(x_test))} mm")
print(f"R²: {r2_score(y_test, best_lgb.predict(x_test))}")

Best Parameters: {'subsample': 0.7, 'num_leaves': 31, 'max_depth': 8, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
RMSE: 116.11274232812701 mm
R²: 0.736320649876574


In [62]:
# train final model
lgb_final = lgb.LGBMRegressor(
    # parameters from tuning
    subsample = 0.7,
    num_leaves = 31,
    n_estimators= 750,
    subsample_freq=1,
    max_depth = 8,
    learning_rate = 0.01,
    colsample_bytree = 0.6,

    # use some parameters from base model
    early_stopping_rounds = 25, 
    random_state = 42,
    n_jobs = -1,
)

# fit model
lgb_final.fit(
    x_tuning, y_tuning,
    eval_X = x_val, eval_y = y_val,
    eval_metric = 'rmse',
        callbacks=[
                lgb.early_stopping(stopping_rounds=25, verbose=False),
                lgb.log_evaluation(period=0)])

# print final statistics
print('Baseline Model:')
print('R²:', lgb_base.score(x_test, y_test))
print(f'RMSE: {root_mean_squared_error(y_test, lgb_base.predict(x_test))} mm')

print('Tuned Model:')
print('R²:', lgb_final.score(x_test, y_test))
print(f'RMSE: {root_mean_squared_error(y_test, lgb_final.predict(x_test))} mm')

Training until validation scores don't improve for 25 rounds
Baseline Model:
R²: 0.7308460886136348
RMSE: 117.31192739575243 mm
Tuned Model:
R²: 0.736320649876574
RMSE: 116.11274232812701 mm


## Step 5: Perform Model Evaluation

This step will calculate RMSE, R², and Kling-Gupta scores for each model, as well as the ensemble mean, and compare them.

R² and RMSE Results:
| ML Model | R² | RMSE | 
| ---- | ---- | ---- |
| Random Forests | 0.737 | 116.060 |
| XGBoost | 0.735 | 116.450 |
| LightGBM | 0.736 | 116.113 |
| Ensemble Mean | 0.740 | 115.281 |

Kling-Gupta Results:
| ML Model | Kling-Gupta Score | r (correlation) | alpha (variance) | beta (bias) | 
| ---- | ---- | ---- | ---- | ---- |
| Random Forests | 0.800 | 0.858 | 0.858 | 0.991 |
| XGBoost | 0.800 | 0.857 | 0.860 | 1.004 |
| LightGBM | 0.808 | 0.858 | 0.870 | 1.010|
| Ensemble Mean | 0.803 | 0.860 | 0.861 | 1.002 |

**Write up of the importance of these stats here**


In [67]:
# define a function to calculate Kling-Gupta scores:

def calculate_kge(y_true, y_pred):
    '''
    Args:
    -----
    y_true (pandas.DataFrame):
        Original data withheld from ML training
    y_pred (pandas.DataFrame):
        Predicted data from trained ML models
    '''
    # Ensure inputs are flat 1D numpy arrays
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    
    # Linear correlation coefficient (r)
    r = np.corrcoef(y_true, y_pred)[0, 1]
    
    # Variability ratio (alpha)
    alpha = np.std(y_pred) / np.std(y_true)
    
    # Bias ratio (beta)
    beta = np.mean(y_pred) / np.mean(y_true)
    
    # Overall KGE score
    kge = 1.0 - np.sqrt((r - 1.0)**2 + (alpha - 1.0)**2 + (beta - 1.0)**2)
    
    return kge, r, alpha, beta

In [77]:
# predict SWE

rf_preds = rf_final.predict(x_test)
xgb_preds = xgb_final.predict(x_test)
lgb_preds = lgb_final.predict(x_test)

In [78]:
# Post-processing physical sanity check (SWE cannot be negative)
rf_preds = np.clip(rf_preds, a_min=0, a_max=None)
xgb_preds = np.clip(xgb_preds, a_min=0, a_max=None)
lgb_preds = np.clip(lgb_preds, a_min=0, a_max=None)

In [79]:
# loop through all models and calculate statistics
models = {
    'Random Forests': rf_preds,
    'XGBoost': xgb_preds,
    'LightGBM': lgb_preds
}

results = []
for name, preds in models.items():
    rmse = root_mean_squared_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    kge, r, alpha, beta = calculate_kge(y_test, preds)
    results.append({
        'ML Model': name,
        'R²': round(r2, 3),
        'RMSE (mm)': round(rmse, 3),
        'Kling-Gupta': round(kge, 3),
        'r (corr)': round(r, 3),
        'alpha (var)': round(alpha, 3),
        'beta (bias)': round(beta, 3)
    })

# Display as a clean comparison table
results_df = pd.DataFrame(results)
print(results_df[['ML Model', 'R²', 'RMSE (mm)', 'Kling-Gupta', 'r (corr)', 'alpha (var)', 'beta (bias)']].to_string(index=False))

      ML Model    R²  RMSE (mm)  Kling-Gupta  r (corr)  alpha (var)  beta (bias)
Random Forests 0.737    116.060        0.800     0.858        0.858        0.991
       XGBoost 0.735    116.446        0.800     0.857        0.860        1.004
      LightGBM 0.736    116.097        0.808     0.858        0.870        1.010


In [81]:
# calculate unweighted ensemble mean prediction
ensemble_preds = np.clip((rf_preds + xgb_preds + lgb_preds) / 3.0, a_min=0, a_max=None)

# evaluate the ensemble against  observations
ens_rmse = root_mean_squared_error(y_test, ensemble_preds)
ens_r2 = r2_score(y_test, ensemble_preds)
ens_kge, ens_r, ens_alpha, ens_beta = calculate_kge(y_test, ensemble_preds)

print("Unweighted Three-Model Ensemble")
print(f"Ensemble R²:          {ens_r2:.3f}")
print(f"Ensemble RMSE:        {ens_rmse:.3f} mm")
print(f"Ensemble KGE:         {ens_kge:.3f}")
print(f"Ensemble Correlation: {ens_r:.3f}")
print(f"Ensemble Alpha (Var): {ens_alpha:.3f}")
print(f"Ensemble Beta (Bias): {ens_beta:.3f}")

Unweighted Three-Model Ensemble
Ensemble R²:          0.740
Ensemble RMSE:        115.281 mm
Ensemble KGE:         0.803
Ensemble Correlation: 0.860
Ensemble Alpha (Var): 0.861
Ensemble Beta (Bias): 1.002


## Step 6: Save models for application to full grids

In [85]:
# make dir for all three models
model_dir = Path(proj_dir, 'trained_models')
os.makedirs(model_dir, exist_ok=True)

# make paths for models
rf_path = Path(model_dir, 'random_forests_swe_model.joblib')
xgb_path = Path(model_dir, 'xgboost_swe_model.joblib')
lgb_path = Path(model_dir, 'lightgbm_swe_model.joblib')

In [86]:
# save models to dir

joblib.dump(rf_final, rf_path, compress = 3)
joblib.dump(xgb_final, xgb_path, compress = 3)
joblib.dump(lgb_final, lgb_path, compress = 3)

['C:\\Users\\raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\trained_models\\lightgbm_swe_model.joblib']

In [87]:
# save feature names for model alignment

# grab names
feature_names = list(x_train.columns)

# make a path
feature_path = Path(model_dir, 'ML_feature_names.joblib')

# save
joblib.dump(feature_names, feature_path)

['C:\\Users\\raini\\Documents\\Graduate_School\\EDA_Certificate\\Summer\\snow-drought-modeling\\trained_models\\ML_feature_names.joblib']